# 🥇 Gold Layer: AAPL Master Intelligence Table
**Purpose:** Explicitly filter for AAPL data, calculate relevance-weighted daily news sentiment, calculate stock performance metrics (with cold-start protections), and join them into a final business-ready table.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
# 1. Configuration
CATALOG = "portfolio"
SCHEMA = "market_data"
SILVER_STOCK_TABLE = f"{CATALOG}.{SCHEMA}.silver_stock_quotes"
SILVER_NEWS_TABLE = f"{CATALOG}.{SCHEMA}.silver_news_sentiment"
GOLD_MASTER_TABLE = f"{CATALOG}.{SCHEMA}.gold_market_intelligence" 

In [0]:
# 2. Read Silver Tables
df_stock_clean = spark.read.table(SILVER_STOCK_TABLE)
df_news_clean = spark.read.table(SILVER_NEWS_TABLE)

In [0]:
# 3. Aggregate Sentiment (Relevance-Weighted Average)
df_daily_sentiment = df_news_clean.groupBy("ticker_symbol", "Date") \
    .agg(
        F.sum(F.col("sentiment_score") * F.col("relevance_score")).alias("sum_weighted_sentiment"),
        F.sum("relevance_score").alias("sum_relevance"),
        F.count("url").alias("article_count")
    ) \
    .withColumn("weighted_avg_sentiment", 
                F.round(F.col("sum_weighted_sentiment") / F.col("sum_relevance"), 3)) \
    .drop("sum_weighted_sentiment", "sum_relevance")

In [0]:
# 4. FULL OUTER JOIN: Combine timelines so we don't drop weekends
df_combined = df_stock_clean.join(
    df_daily_sentiment,
    on=["ticker_symbol", "Date"],
    how="full_outer"
)

In [0]:
# 5. Handle the Weekend/Holiday Gaps (Forward-Fill)
# We use unboundedPreceding to look back at all historical rows and grab the last non-null value
window_ffill = Window.partitionBy("ticker_symbol").orderBy("Date").rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_filled = df_combined \
    .withColumn("is_trading_day", F.when(F.col("Close").isNotNull(), True).otherwise(False)) \
    .withColumn("Close", F.last("Close", ignorenulls=True).over(window_ffill)) \
    .fillna({
        "Volume": 0,                   # No shares traded on weekends
        "weighted_avg_sentiment": 0.0, # Neutral sentiment if no news
        "article_count": 0
    })

In [0]:
# 6. Define Windows for Stock Metrics (Partitioning by ticker just to be safe)
window_spec = Window.partitionBy("ticker_symbol").orderBy("Date")
window_5d = Window.partitionBy("ticker_symbol").orderBy("Date").rowsBetween(-4, 0)

In [0]:
# 7. Calculate Stock Metrics (with cold-start protection)
df_gold_master = df_filled.withColumn("Prev_Close", F.lag("Close").over(window_spec)) \
    .withColumn("Daily_Return_Pct", 
                F.when(F.col("Prev_Close").isNull() | (F.col("Prev_Close") == 0), 0.0)
                 .otherwise(F.round(((F.col("Close") - F.col("Prev_Close")) / F.col("Prev_Close")) * 100, 2))) \
    .withColumn("SMA_5_Day", F.round(F.avg("Close").over(window_5d), 2)) \
    .drop("Prev_Close")

In [0]:
# 7. Write to Gold
df_gold_master.write.format("delta").mode("overwrite").saveAsTable(GOLD_MASTER_TABLE)

display(spark.read.table(GOLD_MASTER_TABLE).orderBy(F.col("Date").desc()).limit(10))

In [0]:
df_gold_master.display()